In [1]:
"""
任务2（优化版）：原始特征与派生特征敏感性实验
优化：去掉特征选择mask → 仅调4个超参；组数7→4；Optuna 50→20
预计: ~1小时 GPU / ~5小时 CPU

组:
  Raw: 仅原始变量+三项土壤变量 (~55维)
  Raw+Cumulative+Delta: Raw + 累积量 + 阶段差分 (~76维)
  Full: 全部100维特征
  Full-AllRatioProxies: Full - 五大比率型代理特征 (~80维)

输出同上。
"""
import pandas as pd
import numpy as np
import os
import warnings
import json
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRFRegressor
from sklearn.model_selection import LeaveOneGroupOut

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ======================== 配置 ========================
DATA_PATH = r"D:\uv_py\xgb\data\P4_Cleaned_Dataset.csv"
OUTPUT_DIR = r"D:\uv_py\xgb\answer_todos\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OPTUNA_TRIALS = 20  # 从50降至20（搜索空间大幅缩小后足够）
RANDOM_SEED = 42
META_COLS = ["Year", "Zone", "latitude", "longitude", "yield"]
STAGES = ["P1", "P2", "P3", "P4"]

# ======================== 特征分组定义 ========================
RAW_METEO = ["Tmean", "PPT", "SM", "GDD", "FDD", "VPD", "VPD_max"]
RAW_RS = ["NDVI", "NDVI_max", "NDWI", "NIRv", "EVI", "EVI_max"]
SOIL = ["Sand", "Clay", "SOC"]
CUM_VARS = ["Cum_PPT", "Cum_FDD", "Cum_VPD"]
DELTA_VARS = ["Delta_NDVI", "Delta_NDWI", "Delta_SM"]
COMPOSITE_KEYWORDS = [
    "WUE", "Decoupling_Stress", "Thermal_Efficiency",
    "Hydrothermal_Balance", "Drought_Vulnerability", "Fertility_Vigor",
]

def build_feature_list(df, group):
    """根据分组名构建特征列表（不含特征选择，直接返回全量）"""
    all_cols = [c for c in df.columns if c not in META_COLS]
    selected = set()

    # 所有组都包含基本变量
    selected.update(SOIL)
    for stage in STAGES:
        for var in RAW_METEO + RAW_RS:
            col = f"{stage}_{var}"
            if col in all_cols:
                selected.add(col)

    # 累积量和差分
    if group in ("raw_cum_delta", "full", "full_all_ratio"):
        for stage in STAGES:
            for var in CUM_VARS:
                col = f"{var}_{stage}"
                if col in all_cols:
                    selected.add(col)
        for stage in ["P2", "P3", "P4"]:
            for var in DELTA_VARS:
                col = f"{var}_{stage}"
                if col in all_cols:
                    selected.add(col)

    # 复合代理特征
    if group in ("full", "full_all_ratio"):
        for stage in STAGES:
            for kw in COMPOSITE_KEYWORDS:
                col = f"{stage}_{kw}"
                if col in all_cols:
                    selected.add(col)

    # full_all_ratio: 删除五大比率型代理特征，仅保留Fertility_Vigor
    if group == "full_all_ratio":
        keep_ratio = {"Fertility_Vigor"}
        for stage in STAGES:
            for kw in COMPOSITE_KEYWORDS:
                if kw not in keep_ratio:
                    selected.discard(f"{stage}_{kw}")

    return sorted(selected)

# ======================== 指标计算 ========================
def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mean_true = np.mean(y_true)
    rrmse = (rmse / mean_true) * 100 if mean_true != 0 else 0
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    den = np.sum((np.abs(y_pred - mean_true) + np.abs(y_true - mean_true)) ** 2)
    d_index = 1 - (np.sum((y_pred - y_true) ** 2) / den) if den != 0 else 0
    return {
        "R2": round(float(r2), 3), "RMSE": round(float(rmse), 3),
        "RRMSE(%)": round(float(rrmse), 3), "MAE": round(float(mae), 3),
        "MAPE(%)": round(float(mape), 3), "d-index": round(float(d_index), 3),
    }

# ======================== MOBO目标函数（简化版：只调超参，不选特征） ========================
def objective(trial, X_np, y_np, groups):
    """仅优化超参数，使用全部特征"""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=50),
        "max_depth": trial.suggest_int("max_depth", 6, 12),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.3, 0.9),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "tree_method": "gpu_hist", "random_state": RANDOM_SEED, "n_jobs": -1,
    }
    logo = LeaveOneGroupOut()
    mae_scores = []
    for train_idx, val_idx in logo.split(X_np, y_np, groups):
        X_tr, X_val = X_np[train_idx], X_np[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        model = XGBRFRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        mae_scores.append(mean_absolute_error(y_val, preds))
    return np.mean(mae_scores), X_np.shape[1]  # 返回特征数（不再优化）

# ======================== 主实验 ========================
GROUPS_TO_RUN = ["raw", "raw_cum_delta", "full", "full_all_ratio"]

print(f"{'='*70}")
print(f">>> 特征敏感性实验（优化版）")
print(f"    搜索方式: 仅超参优化 (4参数), 使用全量特征")
print(f"    特征分组: {GROUPS_TO_RUN}")
print(f"    Optuna寻优次数: {OPTUNA_TRIALS}")
print(f"    总拟合次数: {len(GROUPS_TO_RUN) * 6 * OPTUNA_TRIALS * 4} (约)")
print(f"{'='*70}")

df = pd.read_csv(DATA_PATH)
years = sorted(df["Year"].unique())

all_by_year = []
all_overall = []
all_predictions = []
all_best_params = []

for group in GROUPS_TO_RUN:
    feature_cols = build_feature_list(df, group)
    print(f"\n--- [{group}] 特征数: {len(feature_cols)} ---")

    all_y_true_group, all_y_pred_group = [], []
    fold_results = []

    for test_year in years:
        train_df = df[df["Year"] != test_year]
        test_df = df[df["Year"] == test_year]
        X_train_np = train_df[feature_cols].values
        X_test_np = test_df[feature_cols].values
        y_train_np = train_df["yield"].values
        y_test_np = test_df["yield"].values
        groups_train = train_df["Year"].values
        n_features = X_train_np.shape[1]

        print(f"    [Year {test_year}] Optuna({n_features}feat)...", end="", flush=True)

        study = optuna.create_study(
            directions=["minimize", "minimize"],
            sampler=TPESampler(seed=RANDOM_SEED),
        )
        func = lambda trial: objective(trial, X_train_np, y_train_np, groups_train)
        study.optimize(func, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

        pareto_front = study.best_trials
        best_trial = sorted(pareto_front, key=lambda t: t.values[0])[0]

        best_params = {k: v for k, v in best_trial.params.items()}
        best_params.update({"tree_method": "gpu_hist", "random_state": RANDOM_SEED, "n_jobs": -1})

        final_model = XGBRFRegressor(**best_params)
        final_model.fit(X_train_np, y_train_np)
        y_pred = final_model.predict(X_test_np)

        all_y_true_group.extend(y_test_np)
        all_y_pred_group.extend(y_pred)

        m = calculate_metrics(y_test_np, y_pred)
        m["test_year"] = str(int(test_year))
        m["feature_group"] = group
        m["n_features"] = n_features
        fold_results.append(m)

        all_predictions.append(pd.DataFrame({
            "Year": test_df["Year"].values,
            "Zone": test_df["Zone"].values,
            "yield_true": y_test_np,
            "yield_pred": y_pred,
            "feature_group": group,
        }))

        best_param_record = {"test_year": str(int(test_year)), "feature_group": group}
        for k in ["max_depth", "colsample_bynode", "subsample", "n_estimators"]:
            best_param_record[k] = best_params.get(k, None)
        best_param_record["n_features"] = n_features
        best_param_record["inner_mae"] = float(best_trial.values[0])
        all_best_params.append(best_param_record)

        print(f" done. MAPE={m['MAPE(%)']:.1f}% R2={m['R2']:.3f}")

    global_m = calculate_metrics(all_y_true_group, all_y_pred_group)
    global_m["test_year"] = "Overall"
    global_m["feature_group"] = group
    global_m["n_features"] = len(feature_cols)
    fold_results.append(global_m)
    all_overall.append(global_m)

    print(f"  [{group}] Overall: R2={global_m['R2']:.3f} MAPE={global_m['MAPE(%)']:.1f}%")
    all_by_year.extend([r for r in fold_results if r["test_year"] != "Overall"])

# ======================== 导出 ========================
pd.DataFrame(all_by_year).to_csv(os.path.join(OUTPUT_DIR, "derived_feature_sensitivity_by_year.csv"), index=False)
pd.DataFrame(all_overall).to_csv(os.path.join(OUTPUT_DIR, "derived_feature_sensitivity_overall.csv"), index=False)
pd.concat(all_predictions, ignore_index=True).to_csv(
    os.path.join(OUTPUT_DIR, "derived_feature_sensitivity_predictions.csv"), index=False)
pd.DataFrame(all_best_params).to_csv(os.path.join(OUTPUT_DIR, "derived_feature_sensitivity_best_params.csv"), index=False)

print(f"\n{'='*70}")
print(">>> 敏感性实验完成")
print("    所有输出在:", OUTPUT_DIR)
print(f"{'='*70}")


>>> 特征敏感性实验（优化版）
    搜索方式: 仅超参优化 (4参数), 使用全量特征
    特征分组: ['raw', 'raw_cum_delta', 'full', 'full_all_ratio']
    Optuna寻优次数: 20
    总拟合次数: 1920 (约)

--- [raw] 特征数: 55 ---
    [Year 2016] Optuna(55feat)...

  0%|          | 0/20 [00:00<?, ?it/s]

 done. MAPE=8.8% R2=0.200
    [Year 2017] Optuna(55feat)...

  0%|          | 0/20 [00:00<?, ?it/s]

 done. MAPE=11.0% R2=0.263
    [Year 2018] Optuna(55feat)...

...

 done. MAPE=9.7% R2=0.337
    [Year 2020] Optuna(80feat)...

  0%|          | 0/20 [00:00<?, ?it/s]

 done. MAPE=9.4% R2=0.233
    [Year 2021] Optuna(80feat)...

  0%|          | 0/20 [00:00<?, ?it/s]

 done. MAPE=9.2% R2=0.400
  [full_all_ratio] Overall: R2=0.286 MAPE=9.8%

>>> 敏感性实验完成
    所有输出在: D:\uv_py\xgb\answer_todos\outputs
